## 一、数学原理

### 1. 卷积运算与特征提取

* **一维/二维卷积定义**

  * 一维卷积：

    $$
      (f * g)[i] = \sum_{k=-K}^{K} f[i+k]\;g[k]
    $$
  * 二维卷积（图像）：

    $$
      (I * K)[x,y] = \sum_{u=-m}^{m}\sum_{v=-n}^{n} I[x+u,\,y+v]\;K[u,v]
    $$
* **卷积核（Kernel）**

  * 大小 $5\times5$，参数共享、稀疏连接
  * 输出通道数：卷积核组数
* **激活函数**

  * Tanh：$\sigma(z)=\tanh(z)=\frac{e^z - e^{-z}}{e^z + e^{-z}}$，梯度$1-\tanh^2(z)$
  * ReLU：$\sigma(z)=\max(0,z)$，梯度为 0 或 1

### 2. 池化（下采样）

* **平均池化**

  $$
    y[i,j] = \frac{1}{k^2}\sum_{u=0}^{k-1}\sum_{v=0}^{k-1} x[s\,i+u,\;s\,j+v]
  $$
* **作用**：降低特征图尺寸、增加平移不变性

### 3. 批归一化（Batch Normalization）

* **动机**：减轻内部协变量偏移，加速收敛
* **公式**（对每个通道）

  $$
    \mu_B = \frac1m\sum_{i=1}^m x_i,\quad
    \sigma_B^2 = \frac1m\sum_{i=1}^m (x_i - \mu_B)^2
  $$

  $$
    \hat x_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}},\quad
    y_i = \gamma\,\hat x_i + \beta
  $$
* **可学习参数**：$\gamma$（缩放）、$\beta$（平移）

### 4. Dropout 正则化

* **动机**：防止神经元共适应，等价于多个子网络集成
* **训练时**：每个神经元以概率 $p$ 被“丢弃”
* **测试时**：按 $(1-p)$ 缩放输出


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import copy

# ——— 导入之前定义的模型 ———
from model import LeNet5, LeNet5_Reg

## 二、模型实现流程

### 1. 输入与预处理

* 输入 MNIST 图像 $\,1\times28\times28$
* **Padding** 到 $32\times32$（LeNet-5 原始结构）
* **Normalization**：像素值归一化到 $[0,1]$ 或 $[-1,1]$

### 2. LeNet-5 结构图示

```
[ Input 1×32×32 ]
     ↓ Conv5×5, 6 → [6×28×28] → Tanh → AvgPool2,2 → [6×14×14]
     ↓ Conv5×5,16 → [16×10×10] → Tanh → AvgPool2,2 → [16×5×5]
     ↓ Conv5×5,120 → [120×1×1] → Tanh
     ↓ Flatten → [120]
     ↓ FC120→84 → Tanh
     ↓ FC84→10 → 输出 logits
```

### 3. 加入正则化／归一化变体

* **BatchNorm**：插在每个 Conv→激活 之间
* **ReLU**：替代 Tanh，加速、稳定
* **Dropout**：插在全连接层之前
- **为什么LeNet用Tanh？**
  在1990年代，Tanh是首选激活函数：
  - 解决梯度消失问题优于Sigmoid
  - 对称性有助于网络更快收敛
  - 适合处理数值范围在-1到1之间的数据
- **现代替代品：ReLU**
  现在更常用ReLU（Rectified Linear Unit）：
  ```python
  ReLU(x) = max(0, x)
  ```
  - 优点：计算简单，缓解梯度消失
  - 缺点：负值完全丢失（"死亡ReLU"问题）

In [3]:
# ——— 训练与评估函数 ———
def train_one_epoch(model, device, dataloader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)
    return running_loss / len(dataloader.dataset)

def evaluate(model, device, dataloader, criterion):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            total_loss += criterion(out, y).item() * x.size(0)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
    avg_loss = total_loss / len(dataloader.dataset)
    acc = correct / len(dataloader.dataset)
    return avg_loss, acc

## 三、课堂实验与对比

1. **基础版 vs. Reg 版**

   * 比较收敛速度（loss 曲线）、验证准确率
2. **超参数敏感性**

   * 不同 Dropout 比例（0.2, 0.5, 0.8）对结果影响
3. **Batch 大小**

   * 小 batch（16） vs. 大 batch（128）下 BatchNorm 效果差异


In [2]:
# ——— 主函数：遍历所有实验配置 ———
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 通用超参
    epochs = 10
    lr = 0.01
    momentum = 0.9

    # 实验要比较的设置
    dropout_rates = [0.0, 0.2, 0.5, 0.8]        # 0.0 即基础版 LeNet-5
    batch_sizes   = [16, 128]

    # 数据预处理：MNIST → 32×32, 归一化
    transform = transforms.Compose([
        transforms.Pad(2),  # 28→32
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    train_ds = datasets.MNIST("./data", train=True,  download=True, transform=transform)
    test_ds  = datasets.MNIST("./data", train=False, download=True, transform=transform)

    criterion = nn.CrossEntropyLoss()

    # 依次跑每个实验
    for p in dropout_rates:
        for bs in batch_sizes:
            # 根据 p 选择模型
            if p == 0.0:
                model = LeNet5(num_classes=10)
                exp_name = f"LeNet5_baseline_bs{bs}"
            else:
                model = LeNet5_Reg(num_classes=10, p_dropout=p)
                exp_name = f"LeNet5_Reg_p{p}_bs{bs}"

            model = model.to(device)
            optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)

            train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=2)
            test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False, num_workers=2)

            # 记录每 epoch 的 loss/acc
            history = {"train_loss": [], "val_loss": [], "val_acc": []}

            print(f"\n=== Experiment: {exp_name} ===")
            for epoch in range(1, epochs+1):
                train_loss = train_one_epoch(model, device, train_loader, optimizer, criterion)
                val_loss, val_acc = evaluate(model, device, test_loader, criterion)

                history["train_loss"].append(train_loss)
                history["val_loss"].append(val_loss)
                history["val_acc"].append(val_acc)

                print(f"Epoch {epoch:02d}: "
                      f"train_loss={train_loss:.4f}, "
                      f"val_loss={val_loss:.4f}, "
                      f"val_acc={val_acc*100:.2f}%"
                     )

            # 实验结束，保存模型与日志
            torch.save(model.state_dict(), f"{exp_name}.pth")
            # 你可以根据需要，将 history 序列化成 JSON/CSV 以便后续绘图

In [4]:
if __name__ == "__main__":
    main()


=== Experiment: LeNet5_baseline_bs16 ===
Epoch 01: train_loss=0.1918, val_loss=0.0577, val_acc=98.18%
Epoch 02: train_loss=0.0587, val_loss=0.0483, val_acc=98.48%
Epoch 03: train_loss=0.0428, val_loss=0.0427, val_acc=98.63%
Epoch 04: train_loss=0.0319, val_loss=0.0449, val_acc=98.61%
Epoch 05: train_loss=0.0240, val_loss=0.0391, val_acc=98.76%
Epoch 06: train_loss=0.0212, val_loss=0.0341, val_acc=99.02%
Epoch 07: train_loss=0.0161, val_loss=0.0327, val_acc=98.97%
Epoch 08: train_loss=0.0122, val_loss=0.0343, val_acc=98.99%
Epoch 09: train_loss=0.0104, val_loss=0.0311, val_acc=99.02%
Epoch 10: train_loss=0.0093, val_loss=0.0355, val_acc=99.04%

=== Experiment: LeNet5_baseline_bs128 ===
Epoch 01: train_loss=0.6491, val_loss=0.2088, val_acc=93.77%
Epoch 02: train_loss=0.1558, val_loss=0.1013, val_acc=97.06%
Epoch 03: train_loss=0.0938, val_loss=0.0731, val_acc=97.84%
Epoch 04: train_loss=0.0705, val_loss=0.0578, val_acc=98.19%
Epoch 05: train_loss=0.0584, val_loss=0.0460, val_acc=98.61%


# **性能对比总表**
| 配置                | 最佳验证准确率 | 最终训练损失 | 收敛速度 | 过拟合风险 |
|---------------------|----------------|--------------|----------|------------|
| baseline_bs16       | 99.04%         | 0.0093       | 最快     | 中         |
| baseline_bs128      | 98.92%         | 0.0323       | 中等     | 低         |
| Reg_p0.2_bs16      | **99.30%**     | 0.0207       | 快       | 最低       |
| Reg_p0.2_bs128     | 99.28%         | 0.0329       | 中等     | 低         |
| Reg_p0.5_bs16      | 99.13%         | 0.0457       | 中等     | 低         |
| Reg_p0.5_bs128     | 99.11%         | 0.0598       | 慢       | 最低       |
| Reg_p0.8_bs16      | 89.23%         | 0.6737       | 不收敛   | 高         |
| Reg_p0.8_bs128     | 98.70%         | 0.3196       | 最慢     | 中         |

---
